# Description:

- Create CSV and accompanying plots of appointments for a given scenario

- First of all calculate appointments required by each staff type & appointment mode
{delivered, missed, unmet demand}

- Total demand = Sum{delivered, missed, unmet demand}

- FTE Calculations are in the repo.

 

Per month

Per alliance

Exam question/Objective:
Need the appointments for each of the healthcare staff mentioned.

For each demand scenario

For each capacity scenario

As we will be doing multiple ‘runs’ per scenario we will need mean, median columns. Consider other useful metrics to report confidence.

Key Data Sources:
Output from simulation

This is a very large dictionary

Consider writing a function to break it down.

Methodology/Approach:
Using the met demand + unmet demand, determine how much FTE is required.

Subtasks:
Wrangle data from simulation output

Outputs:
Github branch

CSV

forecast yaml file, run for all regression models

## Library Imports

In [1]:
import os
from pathlib import Path
if 'notebooks' in str(Path.cwd()):
    os.chdir('..')

# Library imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from matplotlib.ticker import FuncFormatter
import datetime as dt
import pickle

# project imports from src
from src.schemas import DataCatalog
from src.various_methods import PlotCounter
from src import constants

# Importing SNEE styles
from sneeifstyles import mpl_style
mpl_style()

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

## Initial Set-up

In [2]:
## Constants
SNEE_SUB_ICB = ['06L','06T','07K']
SNEE_SUB_ICB_NAMES = ['Ipswich & East Suffolk', 'North East Essex', 'West Suffolk']
NOTEBOOK_ALIAS = "Appointments"
GP_PATIENTS_LIST_CATALOG = 'Patients Registered at a GP practice, September 2024'
ONS_POPULATION_CATALOG = 'ONS Population projections'
GP_LIST_AGE_BANDS = constants.GP_LIST_AGE_BANDS # age bands for GP list
GP_LIST_AGE_LABELS = constants.GP_LIST_AGE_LABELS # labels for GP list
SNEE_SUBICB_CODES = list(constants.ONS_CODES.keys()) # list of sub-icb codes for SNEE

# Loading the Data Catalog
catalog =  DataCatalog.load_from_yaml("data_catalog.yaml")

# Initializing the plotCounter object
plot_counter = PlotCounter(name=NOTEBOOK_ALIAS)

# set up output directories
for i in ['outputs/assumptions', 'outputs/plots', 'outputs/tables']:
    if not os.path.exists(i):
        os.makedirs(i)     

## 1. Loading the GP LIST

In [3]:
gp_list_df = catalog.get_catalog_entry_by_name(GP_PATIENTS_LIST_CATALOG)
patients_df = gp_list_df.load()
patients_df.head()

/workspaces/PrimaryCareDemandAndCapacity/src/schemas.py:143: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(csv_file, **read_csv_kwargs)


,PUBLICATION,EXTRACT_DATE,ORG_TYPE,ORG_CODE,ONS_CODE,POSTCODE,SEX,AGE_GROUP_5,NUMBER_OF_PATIENTS
0,GP_PRAC_PAT_LIST,2024-09-01,Comm Region,Y56,E40000003,NaN,ALL,ALL,11051350
1,GP_PRAC_PAT_LIST,2024-09-01,Comm Region,Y56,E40000003,NaN,FEMALE,0_4,255861
2,GP_PRAC_PAT_LIST,2024-09-01,Comm Region,Y56,E40000003,NaN,FEMALE,10_14,300274
3,GP_PRAC_PAT_LIST,2024-09-01,Comm Region,Y56,E40000003,NaN,FEMALE,15_19,295834
4,GP_PRAC_PAT_LIST,2024-09-01,Comm Region,Y56,E40000003,NaN,FEMALE,20_24,384423


### 1.1. Data Processing

- Renames age bands and columns to match ONS projections
- Removing 'all ages' rows
- Dropping unused columns
- Add column with 5 year age banding
- Filter to only SNEE ICB areas
- Group by sum on 5 year age group

In [4]:
#Replacing AGE_BANDS values to match the ONS_projection data
def rename_gp_list_age_bands_and_columns(df):
    """
    Renames age bands and columns in the given DataFrame to match ONS projections.
    Args:
        df (pandas.DataFrame): The DataFrame containing the data to be processed.
    Returns:
        pandas.DataFrame: A copy of the input DataFrame with the age bands and columns renamed.
    """
    df_ = df.copy()
    # replace these specific cases
    df_['AGE_GROUP_5'] = df_['AGE_GROUP_5'].replace({'90_94':'90+', '95+':'90+'})
    # change _ to - for consistency with ONS projections
    df_['AGE_GROUP_5'] = df_['AGE_GROUP_5'].str.replace('_','-')
    #Renaming the column name to match with ONS projection data
    df_ = df_.rename(columns={'ONS_CODE':'AREA_CODE'})
    return df_
    
    
def filter_gp_list_and_drop_unused_columns(df):
    """
    Filters the given DataFrame by dropping unused columns and rows corresponding to 'ALL' for AGE_GROUP_5. 
    Args:
        df (pandas.DataFrame): The input DataFrame to be filtered. 
    Returns:
        pandas.DataFrame: The filtered DataFrame with dropped columns and rows.
    """
    df_ = df.copy()
    # Dropping unused columns and rows corresponding to 'ALL' for AGE_GROUP_5 
    df_ = df_.loc[df_['AGE_GROUP_5']!='ALL']
    # This is due to the 'ALL' row being a sum of all the other rows, the 'SEX' == ALL rows are also dropped
    df_ = df_.drop(columns=['PUBLICATION','EXTRACT_DATE','ORG_TYPE','POSTCODE','SEX','ORG_CODE'])
    return df_


def filter_snee_icb_areas(df):
    """
    Filters the given DataFrame based on the 'ORG_CODE' column, keeping only the rows
    where the 'ORG_CODE' is in the list of SNEE_SUBICB_CODES.
    Args:
        df (pandas.DataFrame): The DataFrame to be filtered.
    Returns:
        pandas.DataFrame: The filtered DataFrame.
    """
    df_ = df.copy()
    df_ = df_.loc[df_['AREA_CODE'].isin(SNEE_SUBICB_CODES)]
    return df_


def groupbysum_area_and_age_group_and_rename_ons_codes(df):
    """
    Groups the DataFrame by AREA_CODE and AGE_GROUP_5, sums the values, drops the AGE_GROUP column if present,
    and renames the index using ONS_CODES.
    Args:
        df (pandas.DataFrame): The input DataFrame.
    Returns:
        pandas.DataFrame: The modified DataFrame with grouped, summed, and renamed values.
    """
    df_ = df.copy()
    # Grouping by AREA_CODE and AGE_GROUP_5 and summing the values
    df_ = df_.groupby(['AREA_CODE','AGE_GROUP_5'], observed=False).sum()
    # Renaming ONS codes with Actual sub icb names
    df_ = df_.rename(index=constants.ONS_CODES)
    return df_

patients_df = (patients_df
                .pipe(rename_gp_list_age_bands_and_columns)
                .pipe(filter_gp_list_and_drop_unused_columns)
                .pipe(filter_snee_icb_areas)
                .pipe(groupbysum_area_and_age_group_and_rename_ons_codes))

patients_df.head()


NUMBER_OF_PATIENTS
AREA_CODE              AGE_GROUP_5                    
Ipswich & East Suffolk 0-4                       17916
                       10-14                     24285
                       15-19                     24206
                       20-24                     19972
                       25-29                     22902

In [5]:
# Transposing the dataframe
patients_df_transformed = (patients_df.T).stack(level='AREA_CODE').reset_index()
patients_df_transformed = patients_df_transformed.drop(columns = ['level_0'])
patients_df_transformed.columns.name = None

patients_df_transformed

,AREA_CODE,0-4,10-14,15-19,20-24,25-29,30-34,35-39,40-44,45-49,5-9,50-54,55-59,60-64,65-69,70-74,75-79,80-84,85-89,90+
0,Ipswich & East Suffolk,17916,24285,24206,19972,22902,26221,27633,26900,24855,21883,27644,29238,28203,24598,22245,21906,13725,8680,5001
1,North East Essex,17157,22233,20284,21635,24086,25674,25041,23187,21337,20395,23223,24945,23766,20572,19178,20181,12611,7486,4229
2,West Suffolk,12253,15571,14965,12841,15341,17696,18210,17584,16356,14182,18652,20623,19385,16073,15151,15560,10047,5913,3093


## 2. Loading ONS Population projections

In [6]:
# Loading the Principal projection
ons_principal_projection = catalog.get_scenario_catalog_entry_by_name(ONS_POPULATION_CATALOG,'Principal projection')
principal_projections_df = ons_principal_projection.load()

# Loading the 10 year migration Projections
ons_10_mig_projection = catalog.get_scenario_catalog_entry_by_name(ONS_POPULATION_CATALOG,'10 year migration variant')
mig_10yr_projections_df = ons_10_mig_projection.load()

# Loading the High international migration Projections
ons_high_mig_projection = catalog.get_scenario_catalog_entry_by_name(ONS_POPULATION_CATALOG,'High international migration variant')
high_mig_projections_df = ons_high_mig_projection.load()

# Loading the Low international migration Projections
ons_low_mig_projection = catalog.get_scenario_catalog_entry_by_name(ONS_POPULATION_CATALOG,'Low international migration variant')
low_mig_projections_df = ons_low_mig_projection.load()

# Loading the Alternative internal migration Projections
ons_alt_mig_projection = catalog.get_scenario_catalog_entry_by_name(ONS_POPULATION_CATALOG,'Alternative internal migration variant')
alt_mig_projections_df = ons_alt_mig_projection.load()

### 2.1. Data Processing
- Removing 'all ages' rows
- removing columns prior to 2024
- removing 'AREA NAME' 'COMPONENT' and 'SEX' column
- Add column with 5 year age banding
- Filter to only SNEE ICB areas
- Group by sum on 5 year age group

In [7]:
def drop_all_ages_and_unused_cols(df) -> pd.DataFrame:
    """
    Rename columns in the DataFrame and drop unused columns.
    Parameters:
    df (pd.DataFrame): The input DataFrame.
    Returns:
    pd.DataFrame: The modified DataFrame with renamed and dropped columns.
    """
    df_ = df.copy()
    df_ = df_.loc[df_['AGE_GROUP'] != 'All ages']
    df_ = df_.drop(columns=['AREA_NAME', 'COMPONENT', 'SEX', '2018', '2019', '2020', '2021', '2022','2023'])
    return df_


def convert_ons_projection_to_5yr_bins(df):
    """
    Convert the ONS projection data to 5-year age bands.
    Args:
        df (pandas.DataFrame): The input DataFrame containing the ONS projection data.
    Returns:
        pandas.DataFrame: The DataFrame with the ONS projection data converted to 5-year age bands.
    """
    df_ = df.copy()

    # replace the text '90 and over' with '90' to match the GP list data
    df_['AGE_GROUP'] = df_['AGE_GROUP'].astype(str).str.replace('90 and over','90')
    #Converting AGE_GROUP data type to int
    df_['AGE_GROUP'] = df_['AGE_GROUP'].astype(int)
    #Adding the column for Age Bands as 'AGE_GROUP_5'
    df_['AGE_GROUP_5'] = pd.cut(df_['AGE_GROUP'], bins=GP_LIST_AGE_BANDS, labels=GP_LIST_AGE_LABELS, include_lowest=True, right=False, ordered=True)
    # Dropping the original age column
    df_ = df_.drop(columns=['AGE_GROUP'], errors='ignore')
    return df_


def groupbysum_area_and_age_group_and_rename_ons_codes(df):
    """
    Groups the DataFrame by AREA_CODE and AGE_GROUP_5, sums the values, drops the AGE_GROUP column if present,
    and renames the index using ONS_CODES.
    Args:
        df (pandas.DataFrame): The input DataFrame.
    Returns:
        pandas.DataFrame: The modified DataFrame with grouped, summed, and renamed values.
    """
    df_ = df.copy()
    # Grouping by AREA_CODE and AGE_GROUP_5 and summing the values
    df_ = df_.groupby(['AREA_CODE','AGE_GROUP_5'], observed=False).sum()
    # Renaming ONS codes with Actual sub icb names
    df_ = df_.rename(index=constants.ONS_CODES)
    return df_


principal_projections_df = (principal_projections_df
                            .pipe(drop_all_ages_and_unused_cols)
                            .pipe(convert_ons_projection_to_5yr_bins)
                            .pipe(filter_snee_icb_areas)
                            .pipe(groupbysum_area_and_age_group_and_rename_ons_codes))
principal_projections_df.head()

2024       2025       2026  \
AREA_CODE              AGE_GROUP_5                                    
Ipswich & East Suffolk 0-4          19698.373  19604.966  19527.257   
                       5-9          22378.224  21998.493  21529.082   
                       10-14        25016.257  24783.209  24569.717   
                       15-19        23496.655  23897.784  24193.275   
                       20-24        17173.888  17191.096  17520.013   

                                         2027       2028       2029  \
AREA_CODE              AGE_GROUP_5                                    
Ipswich & East Suffolk 0-4          19483.805  19446.523  19414.341   
                       5-9          21261.284  21050.186  20983.672   
                       10-14        24249.751  24005.345  23584.788   
                       15-19        24317.998  24191.858  24085.124   
                       20-24        17987.881  18489.898  18808.913   

                                         2030       2031       2032  \
AREA_CODE              AGE_GROUP_5                                    
Ipswich & East Suffolk 0-4          19391.237  19381.836  19393.561   
                       5-9          20896.032  20825.220  20787.879   
                       10-14        23208.398  22734.049  22463.455   
                       15-19        23861.980  23735.101  23447.633   
                       20-24        19207.344  19504.800  19722.030   

                                         2033       2034       2035  \
AREA_CODE              AGE_GROUP_5                                    
Ipswich & East Suffolk 0-4          19429.657  19490.582  19575.604   
                       5-9          20756.586  20730.435  20713.191   
                       10-14        22247.084  22183.313  22097.687   
                       15-19        23183.517  22773.296  22376.525   
                       20-24        19737.583  19789.355  19742.953   

                                         2036       2037       2038  \
AREA_CODE              AGE_GROUP_5                                    
Ipswich & East Suffolk 0-4          19686.005  19821.500  19978.838   
                       5-9          20709.948  20728.187  20771.506   
                       10-14        22028.982  21994.096  21965.440   
                       15-19        21998.140  21733.338  21566.061   
                       20-24        19656.351  19404.588  19195.243   

                                         2039       2040       2041  \
AREA_CODE              AGE_GROUP_5                                    
Ipswich & East Suffolk 0-4          20152.670  20332.876  20507.740   
                       5-9          20840.531  20934.368  21054.439   
                       10-14        21942.197  21928.291  21928.799   
                       15-19        21499.806  21422.362  21367.768   
                       20-24        18928.754  18660.077  18324.626   

                                         2042       2043  
AREA_CODE              AGE_GROUP_5                        
Ipswich & East Suffolk 0-4          20666.628  20800.504  
                       5-9          21200.433  21369.098  
                       10-14        21951.560  22000.292  
                       15-19        21336.679  21311.481  
                       20-24        18080.678  17917.411

### 2.2. Calculating the population change factor for each year and tranposing the dataframe
- 2024 as baseline

In [8]:
def calculate_population_change_factor(df):
    """
    Calculates the population change factor by keeping 2024 population as baseline and dividing all years population by the baseline
    Args:
        df (pandas.DataFrame): The input DataFrame.
    Returns:
        pandas.DataFrame: The modified DataFrame with population change factors for each year
    """
    df_ = df.copy()
    # create a baseline (series/1 column)
    pop_baseline:pd.Series = df_['2024']
    # divide the projections by the baseline to get the relative factors
    df_:pd.DataFrame = df_.div(pop_baseline, axis=0).round(5)
    return df_


def transpose_dataframe(df):
    df_ = df.copy()
    # TRansposing the dataframe
    df_ = (df_.T).stack(level='AREA_CODE').reset_index().rename(columns={'level_0': 'DATE'})
    df_.columns.name = None
    df_ = df_.sort_values(by=['AREA_CODE', 'DATE']).reset_index(drop=True)
    # Adding month and date in the DATE column
    df_['DATE'] = df_['DATE'].str.zfill(4) + '-09-01'
    # Converting DATE back to datetime
    df_['DATE'] = pd.to_datetime(df_['DATE'])
    return df_


principal_projections_df_ = (principal_projections_df
                            .pipe(calculate_population_change_factor)
                            .pipe(transpose_dataframe))
principal_projections_df_.head()

,DATE,AREA_CODE,0-4,5-9,10-14,15-19,20-24,25-29,30-34,35-39,...,45-49,50-54,55-59,60-64,65-69,70-74,75-79,80-84,85-89,90+
0,2024-09-01,Ipswich & East Suffolk,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,...,1.00000,1.00000,1.00000,1.00000,1.0000,1.00000,1.00000,1.00000,1.00000,1.00000
1,2025-09-01,Ipswich & East Suffolk,0.99526,0.98303,0.99068,1.01707,1.00100,0.98387,0.99131,1.00265,...,1.01413,0.96956,0.99293,1.03106,1.0204,0.99384,1.01966,1.05365,1.01015,1.03269
2,2026-09-01,Ipswich & East Suffolk,0.99131,0.96205,0.98215,1.02965,1.02015,0.96232,0.97853,1.00799,...,1.03579,0.92896,0.99006,1.04575,1.0540,0.99569,1.02749,1.12172,1.01096,1.05683
3,2027-09-01,Ipswich & East Suffolk,0.98911,0.95009,0.96936,1.03496,1.04740,0.93405,0.97215,1.00579,...,1.06271,0.89626,0.97767,1.06185,1.0723,1.01661,0.97600,1.25321,1.04241,1.08529
4,2028-09-01,Ipswich & East Suffolk,0.98721,0.94065,0.95959,1.02959,1.07663,0.91783,0.96578,0.99540,...,1.08015,0.86974,0.96606,1.06365,1.0975,1.04052,0.95357,1.32208,1.08616,1.12230


### 2.3. Making synthetic data using interpolation

In [9]:
date_range_df = pd.DataFrame(pd.date_range(start=dt.date(year=int(principal_projections_df.T.index[0]),month=9,day=1), end=dt.date(year=int(principal_projections_df.T.index[-1]),month=9,day=1), freq='MS')).rename(columns={0:'DATE'})
date_range_df = date_range_df.loc[date_range_df.index.repeat(3)]
date_range_df['AREA_CODE'] = SNEE_SUB_ICB_NAMES * (len(date_range_df) // len(SNEE_SUB_ICB_NAMES))

date_range_df

,DATE,AREA_CODE
0,2024-09-01,Ipswich & East Suffolk
0,2024-09-01,North East Essex
0,2024-09-01,West Suffolk
1,2024-10-01,Ipswich & East Suffolk
1,2024-10-01,North East Essex
...,...,...
227,2043-08-01,North East Essex
227,2043-08-01,West Suffolk
228,2043-09-01,Ipswich & East Suffolk
228,2043-09-01,North East Essex


In [10]:
interpolated_df = pd.merge(date_range_df, principal_projections_df_, on=('DATE','AREA_CODE'), how='left')
interpolated_df = interpolated_df.sort_values(by=['AREA_CODE', 'DATE']).reset_index(drop=True)
interpolated_df = interpolated_df.interpolate(method='linear')
interpolated_df = interpolated_df.set_index(['AREA_CODE','DATE'])
interpolated_df

/tmp/ipykernel_36974/1603379569.py:3: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  interpolated_df = interpolated_df.interpolate(method='linear')


0-4       5-9     10-14     15-19  \
AREA_CODE              DATE                                                 
Ipswich & East Suffolk 2024-09-01  1.000000  1.000000  1.000000  1.000000   
                       2024-10-01  0.999605  0.998586  0.999223  1.001422   
                       2024-11-01  0.999210  0.997172  0.998447  1.002845   
                       2024-12-01  0.998815  0.995757  0.997670  1.004268   
                       2025-01-01  0.998420  0.994343  0.996893  1.005690   
...                                     ...       ...       ...       ...   
West Suffolk           2043-05-01  1.079570  1.003833  0.930160  0.988413   
                       2043-06-01  1.080232  1.004570  0.930413  0.988335   
                       2043-07-01  1.080895  1.005307  0.930665  0.988257   
                       2043-08-01  1.081557  1.006043  0.930918  0.988178   
                       2043-09-01  1.082220  1.006780  0.931170  0.988100   

                                      20-24     25-29     30-34     35-39  \
AREA_CODE              DATE                                                 
Ipswich & East Suffolk 2024-09-01  1.000000  1.000000  1.000000  1.000000   
                       2024-10-01  1.000083  0.998656  0.999276  1.000221   
                       2024-11-01  1.000167  0.997312  0.998552  1.000442   
                       2024-12-01  1.000250  0.995968  0.997827  1.000663   
                       2025-01-01  1.000333  0.994623  0.997103  1.000883   
...                                     ...       ...       ...       ...   
West Suffolk           2043-05-01  1.084883  1.055337  1.046260  0.932887   
                       2043-06-01  1.084217  1.054575  1.046825  0.934987   
                       2043-07-01  1.083552  1.053813  1.047390  0.937088   
                       2043-08-01  1.082886  1.053052  1.047955  0.939189   
                       2043-09-01  1.082220  1.052290  1.048520  0.941290   

                                      40-44     45-49     50-54     55-59  \
AREA_CODE              DATE                                                 
Ipswich & East Suffolk 2024-09-01  1.000000  1.000000  1.000000  1.000000   
                       2024-10-01  1.000042  1.001178  0.997463  0.999411   
                       2024-11-01  1.000085  1.002355  0.994927  0.998822   
                       2024-12-01  1.000128  1.003532  0.992390  0.998232   
                       2025-01-01  1.000170  1.004710  0.989853  0.997643   
...                                     ...       ...       ...       ...   
West Suffolk           2043-05-01  0.927733  1.097987  0.965283  0.892017   
                       2043-06-01  0.926645  1.097647  0.964145  0.893112   
                       2043-07-01  0.925557  1.097308  0.963007  0.894208   
                       2043-08-01  0.924468  1.096969  0.961868  0.895304   
                       2043-09-01  0.923380  1.096630  0.960730  0.896400   

                                      60-64     65-69     70-74     75-79  \
AREA_CODE              DATE                                                 
Ipswich & East Suffolk 2024-09-01  1.000000  1.000000  1.000000  1.000000   
                       2024-10-01  1.002588  1.001700  0.999487  1.001638   
                       2024-11-01  1.005177  1.003400  0.998973  1.003277   
                       2024-12-01  1.007765  1.005100  0.998460  1.004915   
                       2025-01-01  1.010353  1.006800  0.997947  1.006553   
...                                     ...       ...       ...       ...   
West Suffolk           2043-05-01  0.938943  1.037510  1.250093  1.166637   
                       2043-06-01  0.940608  1.035235  1.248685  1.167060   
                       2043-07-01  0.942272  1.032960  1.247277  1.167483   
                       2043-08-01  0.943936  1.030685  1.245868  1.167907   
                       2043-09-01  0.945600  1.028410  1.244460  1.168330   

                                

## 3. Multiplying change factors by population from GP list (09/2024)

In [11]:
interpolated_df.loc[:,GP_LIST_AGE_LABELS] = (interpolated_df.loc[:,GP_LIST_AGE_LABELS] * patients_df_transformed.set_index('AREA_CODE')).round(2)

# Calculating the total population
interpolated_df['Total Population'] = interpolated_df.sum(axis=1)
interpolated_df



0-4       5-9     10-14     15-19  \
AREA_CODE              DATE                                                 
Ipswich & East Suffolk 2024-09-01  17916.00  21883.00  24285.00  24206.00   
                       2024-10-01  17908.92  21852.05  24266.14  24240.43   
                       2024-11-01  17901.85  21821.11  24247.28  24274.87   
                       2024-12-01  17894.77  21790.16  24228.42  24309.30   
                       2025-01-01  17887.69  21759.22  24209.55  24343.73   
...                                     ...       ...       ...       ...   
West Suffolk           2043-05-01  13227.97  14236.36  14483.52  14791.61   
                       2043-06-01  13236.09  14246.81  14487.45  14790.43   
                       2043-07-01  13244.21  14257.26  14491.38  14789.26   
                       2043-08-01  13252.32  14267.71  14495.32  14788.09   
                       2043-09-01  13260.44  14278.15  14499.25  14786.92   

                                      20-24     25-29     30-34     35-39  \
AREA_CODE              DATE                                                 
Ipswich & East Suffolk 2024-09-01  19972.00  22902.00  26221.00  27633.00   
                       2024-10-01  19973.66  22871.22  26202.01  27639.10   
                       2024-11-01  19975.33  22840.43  26183.02  27645.20   
                       2024-12-01  19976.99  22809.65  26164.03  27651.31   
                       2025-01-01  19978.66  22778.86  26145.05  27657.41   
...                                     ...       ...       ...       ...   
West Suffolk           2043-05-01  13930.99  16189.92  18514.62  16987.87   
                       2043-06-01  13922.44  16178.24  18524.62  17026.12   
                       2043-07-01  13913.89  16166.55  18534.61  17064.38   
                       2043-08-01  13905.34  16154.87  18544.61  17102.63   
                       2043-09-01  13896.79  16143.18  18554.61  17140.89   

                                      40-44     45-49     50-54     55-59  \
AREA_CODE              DATE                                                 
Ipswich & East Suffolk 2024-09-01  26900.00  24855.00  27644.00  29238.00   
                       2024-10-01  26901.14  24884.27  27573.88  29220.77   
                       2024-11-01  26902.29  24913.53  27503.75  29203.55   
                       2024-12-01  26903.43  24942.80  27433.63  29186.32   
                       2025-01-01  26904.57  24972.07  27363.51  29169.10   
...                                     ...       ...       ...       ...   
West Suffolk           2043-05-01  16313.26  17958.67  18004.46  18396.06   
                       2043-06-01  16294.13  17953.12  17983.23  18418.66   
                       2043-07-01  16274.99  17947.58  17962.00  18441.26   
                       2043-08-01  16255.85  17942.03  17940.77  18463.86   
                       2043-09-01  16236.71  17936.48  17919.54  18486.46   

                                      60-64     65-69     70-74     75-79  \
AREA_CODE              DATE                                                 
Ipswich & East Suffolk 2024-09-01  28203.00  24598.00  22245.00  21906.00   
                       2024-10-01  28276.00  24639.82  22233.58  21941.89   
                       2024-11-01  28349.00  24681.63  22222.16  21977.78   
                       2024-12-01  28422.00  24723.45  22210.74  22013.67   
                       2025-01-01  28495.00  24765.27  22199.32  22049.56   
...                                     ...       ...       ...       ...   
West Suffolk           2043-05-01  18201.42  16675.90  18940.16  18152.87   
                       2043-06-01  18233.68  16639.33  18918.83  18159.45   
                       2043-07-01  18265.94  16602.77  18897.49  18166.04   
                       2043-08-01  18298.20  16566.20  18876.15  18172.63   
                       2043-09-01  18330.46  16529.63  18854.81  18179.21   

                                

In [12]:
monthly_pop:pd.Series = interpolated_df['Total Population']

interpolated_df:pd.DataFrame = interpolated_df.div(monthly_pop, axis=0).round(5).reset_index()
interpolated_df = interpolated_df.drop(columns=['Total Population'])
interpolated_df

,AREA_CODE,DATE,0-4,5-9,10-14,15-19,20-24,25-29,30-34,35-39,...,45-49,50-54,55-59,60-64,65-69,70-74,75-79,80-84,85-89,90+
0,Ipswich & East Suffolk,2024-09-01,0.04286,0.05235,0.05810,0.05791,0.04778,0.05479,0.06273,0.06611,...,0.05946,0.06613,0.06995,0.06747,0.05885,0.05322,0.05241,0.03283,0.02076,0.01196
1,Ipswich & East Suffolk,2024-10-01,0.04283,0.05226,0.05804,0.05798,0.04777,0.05470,0.06267,0.06610,...,0.05952,0.06595,0.06989,0.06763,0.05893,0.05318,0.05248,0.03297,0.02078,0.01199
2,Ipswich & East Suffolk,2024-11-01,0.04281,0.05218,0.05798,0.05804,0.04776,0.05461,0.06261,0.06610,...,0.05957,0.06576,0.06983,0.06779,0.05902,0.05314,0.05255,0.03311,0.02079,0.01202
3,Ipswich & East Suffolk,2024-12-01,0.04278,0.05209,0.05792,0.05811,0.04776,0.05453,0.06255,0.06610,...,0.05963,0.06558,0.06977,0.06794,0.05910,0.05310,0.05262,0.03325,0.02080,0.01205
4,Ipswich & East Suffolk,2025-01-01,0.04275,0.05200,0.05786,0.05818,0.04775,0.05444,0.06249,0.06610,...,0.05968,0.06540,0.06971,0.06810,0.05919,0.05306,0.05270,0.03339,0.02082,0.01208
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
682,West Suffolk,2043-05-01,0.04491,0.04833,0.04917,0.05022,0.04730,0.05497,0.06286,0.05768,...,0.06097,0.06113,0.06246,0.06180,0.05662,0.06431,0.06163,0.04879,0.03028,0.02119
683,West Suffolk,2043-06-01,0.04493,0.04836,0.04918,0.05020,0.04726,0.05491,0.06288,0.05779,...,0.06094,0.06104,0.06252,0.06189,0.05648,0.06422,0.06164,0.04891,0.03033,0.02120
684,West Suffolk,2043-07-01,0.04495,0.04838,0.04918,0.05019,0.04722,0.05486,0.06290,0.05791,...,0.06091,0.06096,0.06258,0.06199,0.05634,0.06413,0.06165,0.04903,0.03039,0.02122
685,West Suffolk,2043-08-01,0.04496,0.04841,0.04918,0.05017,0.04718,0.05481,0.06292,0.05803,...,0.06087,0.06087,0.06264,0.06208,0.05621,0.06404,0.06166,0.04915,0.03045,0.02123


## load model

In [13]:
#model = /workspaces/PrimaryCareDemandAndCapacity/outputs/demographic-month-sklearn.pkl

# Load the model from the .pkl file
with open('/workspaces/PrimaryCareDemandAndCapacity/outputs/demographic-month-sklearn.pkl', 'rb') as file:  # Use the correct path to your .pkl file
    model = pickle.load(file)
    
model

AttributeError: Can't get attribute 'sin_cos_transformer' on <module '__main__'>